# EOM SFT Fine-tuning — Gemma-2-2B on Kaggle T4

**Objective**: Fine-tune a LoRA adapter on the EOM SFT dataset so that a small
local Gemma model can compile source documents into EOM JSON without calling the
OpenRouter API.

## Dataset stats
| Split | Examples |
|-------|----------|
| train (sft.jsonl) | 104 |
| val (val.jsonl) | 5 |
| test (test.jsonl) | 7 |

Token lengths (combined input+target): median 3.8K, p95 8.7K, max 12K.

**max_seq_length is set to 4096** (not 8K) because a T4 (16 GB) with batch=1
and 4-bit Gemma-2-2B is borderline at 8K. Examples exceeding 4K tokens are
filtered out (~10% of train). Extend to 8K if you switch to a GPU with more VRAM.

## Model choice
Default: `unsloth/gemma-2-2b-it-bnb-4bit` — a 4-bit quantised Gemma-2-2B instruction
model pre-packaged by Unsloth for fast LoRA training. You can swap `model_name` to:
- `unsloth/gemma-2-9b-it-bnb-4bit` — for a stronger student (needs A100 or H100)
- `unsloth/gemma-3-1b-it-bnb-4bit` — smaller, faster, lower quality
- Any other Unsloth-packaged 4-bit Gemma variant

## Upload your data to Kaggle
Before running this notebook, attach your JSONL files. Two options:

**Option A — Upload as a Kaggle Dataset**
1. Zip `data/train/sft.jsonl`, `val.jsonl`, `test.jsonl`.
2. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → `+ New Dataset`.
3. Name it `eom-sft`, upload the zip, make it private.
4. In this notebook: `Add Data` → search for `eom-sft` and attach.
5. Files land at `/kaggle/input/datasets/whymelabs/eom-sft/{sft,val,test}.jsonl`.

**Option B — Upload as a Kaggle Notebook dataset (quick)**
1. In the notebook sidebar click `Add Data` → `Upload` → drop the 3 JSONL files.
2. Update the `data_files` paths in Cell 6 to match where Kaggle places them.

## Output
After training the notebook saves:
- `/kaggle/working/eom-sft-adapter/` — the LoRA adapter (download via Kaggle Output tab)
- `/kaggle/working/val-generations.jsonl` — greedy decodes on val split for offline eval

Download the adapter and point `EOM_FINETUNED_CKPT` at it to use `FineTunedCompiler` locally.

In [ ]:
# Install dependencies.
# Unsloth's install matrix is GPU-specific; the simple pip command works on
# Kaggle T4 as of mid-2026. If you see CUDA-related errors, consult
# https://github.com/unslothai/unsloth#installation for the explicit wheel URL.
!pip install -q -U unsloth bitsandbytes peft trl datasets accelerate

In [ ]:
import torch
from unsloth import FastLanguageModel

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# --- Model choice ---
# Default: Gemma-3-4B instruction-tuned, 4-bit. Gemma-3 was designed for fp16
# numerical stability (Gemma-2 in fp16 on T4 had loss explosions: 23.9 in v8).
# 4B params (vs 2B for Gemma-2) is closer to the plan's "Gemma-4 E2B" target
# and fits T4 16GB at 8K context with LoRA + grad-checkpointing.
model_name = "unsloth/gemma-3-4b-it-bnb-4bit"
max_seq_length = 8192  # bumped from 4K — half the dataset was being filtered

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,       # auto-detect (fp16 on T4)
    load_in_4bit=True,
)
print("Model loaded:", model_name)

In [ ]:
# Apply LoRA adapter:
#   rank=16, alpha=32, all attention + MLP projection matrices.
#   gradient_checkpointing="unsloth" saves ~30% VRAM vs standard.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

# --- Data paths ---
# Adjust these if you attached the files under a different Kaggle dataset name.
TRAIN_PATH = "/kaggle/input/datasets/whymelabs/eom-sft/sft.jsonl"
VAL_PATH   = "/kaggle/input/datasets/whymelabs/eom-sft/val.jsonl"

raw = load_dataset("json", data_files={"train": TRAIN_PATH, "val": VAL_PATH})
print("Raw dataset:", {k: len(v) for k, v in raw.items()})


def to_chat(ex):
    """Convert an (input, target) row into a single Gemma chat string.

    The Gemma chat template wraps the user turn in <start_of_turn>user ... <end_of_turn>
    and the model reply in <start_of_turn>model ... <end_of_turn>.
    Both turns are concatenated so the loss is computed over the full sequence;
    in practice TRL's SFTTrainer will mask the prompt tokens when using
    dataset_text_field + a completion-only data collator, but cross-entropy over
    the full sequence is also acceptable for this dataset size.
    """
    messages = [
        {"role": "user",      "content": ex["input"]},
        {"role": "assistant", "content": ex["target"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


dataset = raw.map(to_chat, remove_columns=raw["train"].column_names)


def fits_in_context(ex):
    """Return True if the example fits within max_seq_length tokens.

    Examples exceeding 4K are silently dropped (~10% of train based on p95=8.7K stats).
    This avoids silent truncation which would corrupt the JSON target.
    """
    ids = tokenizer(ex["text"], add_special_tokens=False)["input_ids"]
    return len(ids) <= max_seq_length


dataset = dataset.filter(fits_in_context)
print("After length filter:", {k: len(v) for k, v in dataset.items()})

In [ ]:
from trl import SFTTrainer, SFTConfig

# v9 hit `ImportError: cannot import name 'DataCollatorForCompletionOnlyLM' from
# 'trl'` — the modern TRL (≥0.20) replaces the explicit collator with the
# SFTConfig flag `assistant_only_loss=True`, which masks the prompt tokens
# automatically using the chat template's role markers.

# Effective batch = per_device (1) × grad_accum (16) = 16.
args = SFTConfig(
    output_dir="/kaggle/working/eom-sft-out",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    max_seq_length=max_seq_length,
    packing=False,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    bf16=False,
    seed=42,
    report_to="none",
    assistant_only_loss=True,            # mask prompt; train only on EOM JSON
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    args=args,
)
n_train = len(dataset["train"])
eff_batch = args.per_device_train_batch_size * args.gradient_accumulation_steps
print(f"Steps per epoch: {n_train // eff_batch} (train={n_train}, eff_batch={eff_batch})")

In [ ]:
# --- Training ---
# On a Kaggle T4 with ~100 examples and 3 epochs this takes roughly 15-25 minutes.
# Eval cross-entropy loss is reported once per epoch on the val split.
# No harness evaluation is run here — download the adapter and run the full harness
# locally with `uv run pytest tests/` after setting EOM_FINETUNED_CKPT.
trainer_stats = trainer.train()
print("Training complete.")
print(f"  Peak VRAM: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# --- Validation generations ---
# Generate greedy decodes for each val example and save them so you can run
# the EOM harness offline (the harness requires the `eom` package which is not
# installed in the Kaggle environment).
import json

FastLanguageModel.for_inference(model)  # enables Unsloth's fast inference mode

gens = []
for ex in dataset["val"]:
    # Strip everything from the model turn onwards to build the prompt-only prefix.
    prompt_text = ex["text"].split("<start_of_turn>model")[0] + "<start_of_turn>model\n"
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            temperature=0.0,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    gens.append({"prompt": prompt_text, "generated": generated})

out_path = "/kaggle/working/val-generations.jsonl"
with open(out_path, "w") as f:
    for g in gens:
        f.write(json.dumps(g) + "\n")
print(f"Saved {len(gens)} val generations to {out_path}")

In [ ]:
import os

# --- Save LoRA adapter ---
# Only saves the adapter weights (~30 MB for rank-16 LoRA), not the full model.
# Download via Kaggle's Output tab after the notebook finishes.
ADAPTER_PATH = "/kaggle/working/eom-sft-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

adapter_files = os.listdir(ADAPTER_PATH)
print("Files:", adapter_files)

## Using the adapter locally with FineTunedCompiler

After downloading the adapter from the Kaggle Output tab:

```bash
# 1. Unzip / place the adapter directory somewhere, e.g.:
#    ~/models/eom-sft-adapter/

# 2. Set the env var:
export EOM_FINETUNED_CKPT="/path/to/eom-sft-adapter"

# 3. Use the compiler:
python - <<'EOF'
from eom.compilers.finetuned import FineTunedCompiler
c = FineTunedCompiler()
with open("my_document.md") as f:
    eom = c.compile(f.read(), hints={"document_type": "memo"})
import json
print(json.dumps(eom.model_dump(mode="json"), indent=2))
EOF
```

**Notes:**
- `FineTunedCompiler` uses `transformers` + `peft`, not Unsloth, so it works on
  GPUs without 4-bit support (e.g. 1080 Ti / Pascal) in fp16 mode.
- The base model (`google/gemma-2-2b-it`) is downloaded from Hugging Face on first
  use (~5 GB). Set `HF_HOME` to control the cache location.
- If you fine-tuned a different Gemma variant, pass `base_model_name=` to the
  constructor, e.g. `FineTunedCompiler(base_model_name="google/gemma-2-9b-it")`.
- On generation failure the compiler falls back to `RulesCompiler` automatically.

### Running the harness on val generations

To evaluate against the EOM harness without a GPU, download
`val-generations.jsonl` from the Kaggle Output tab and run:

```bash
# Parse each generated JSON and run the harness:
python scripts/eval_generations.py val-generations.jsonl
```

(Or import `eom.harness` directly if you have the eom package installed.)